In [ ]:
import sys
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install transformers accelerate einops sentencepiece sentence-transformers

In [ ]:
import os
import json
import random
import platform
from dataclasses import dataclass
from typing import List, Dict

import numpy as np
import torch
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer


In [ ]:
@dataclass
class Config:
    model_id: str = "Qwen/Qwen2.5-0.5B-Instruct"
    max_len: int = 256
    max_new_tokens: int = 128
    seed: int = 42
    dtype: torch.dtype = torch.float32
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

cfg = Config()


In [ ]:
random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

print("Device:", cfg.device)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.model_id, use_fast=True)

model = AutoModelForCausalLM.from_pretrained(
    cfg.model_id,
    torch_dtype=cfg.dtype,
    device_map=None
).to(cfg.device)

model.eval()

In [ ]:
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=cfg.device
)

In [ ]:
def get_num_layers(m):
    if hasattr(m, "model") and hasattr(m.model, "layers"):
        return len(m.model.layers)
    raise ValueError("Cannot find transformer layers")

NUM_LAYERS = get_num_layers(model)
print("Num layers:", NUM_LAYERS)

In [ ]:
DIRECT_PROMPTS = [
    "Explain recursion clearly and concisely.",
    "Explain gradient descent clearly.",
    "Explain attention in transformers clearly.",
    "Explain what a compiler does clearly.",
    "Explain what a linked list is clearly.",
    "Explain binary search clearly.",
    "Explain overfitting in machine learning clearly.",
    "Explain backpropagation clearly.",
    "Explain convolutional neural networks clearly.",
    "Explain the bias-variance tradeoff clearly.",
    "Explain what a database index is clearly.",
    "Explain operating system scheduling clearly.",
    "Explain what a hash table is clearly.",
    "Explain floating point precision clearly.",
    "Explain what an API is clearly.",
    "Explain what a deadlock is clearly.",
    "Explain normalization in databases clearly.",
    "Explain what a cache is clearly.",
    "Explain supervised learning clearly.",
    "Explain unsupervised learning clearly."
]


HEDGED_PROMPTS = [
    "Explain recursion, but include caveats, limitations, and cautious language.",
    "Explain gradient descent, but include caveats and assumptions.",
    "Explain attention in transformers cautiously, mentioning limitations.",
    "Explain what a compiler does, with disclaimers and careful wording.",
    "Explain what a linked list is, while noting edge cases and trade-offs.",
    "Explain binary search cautiously and mention when it may not apply.",
    "Explain overfitting with caveats and contextual limitations.",
    "Explain backpropagation cautiously, including assumptions.",
    "Explain convolutional neural networks with disclaimers and limitations.",
    "Explain the bias-variance tradeoff cautiously.",
    "Explain what a database index is, noting trade-offs.",
    "Explain operating system scheduling cautiously.",
    "Explain what a hash table is, including limitations.",
    "Explain floating point precision cautiously and note inaccuracies.",
    "Explain what an API is with disclaimers.",
    "Explain what a deadlock is cautiously.",
    "Explain normalization in databases with caveats.",
    "Explain what a cache is cautiously.",
    "Explain supervised learning cautiously.",
    "Explain unsupervised learning cautiously."
]


EVAL_PROMPTS = [
    "Explain recursion.",
    "Explain gradient.",
    "Explain attention.",
    "Explain what a compiler does.",
    "Explain what a linked list is.",
    "Explain binary search.",
    "Explain overfitting in machine learning.",
    "Explain backpropagation.",
    "Explain convolutional neural networks.",
    "Explain the bias-variance tradeoff.",
    "Explain what a database index is.",
    "Explain operating system scheduling.",
    "Explain what a hash table is.",
    "Explain floating point precision.",
    "Explain what an API is.",
    "Explain what a deadlock is.",
    "Explain normalization in databases.",
    "Explain what a cache is.",
    "Explain supervised learning.",
    "Explain unsupervised learning."
]

In [ ]:
def format_chat(prompt: str) -> str:
    msgs = [{"role": "user", "content": prompt}]
    return tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True
    )

In [ ]:
@torch.no_grad()
def get_last_token_hidden(prompt: str, layer: int) -> torch.Tensor:
    text = format_chat(prompt)
    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=cfg.max_len
    ).to(cfg.device)

    out = model(**enc, output_hidden_states=True, use_cache=False)

    hs = out.hidden_states[layer + 1]
    last_idx = int(enc["attention_mask"][0].sum().item()) - 1
    vec = hs[0, last_idx].detach().to("cpu")
    return vec

In [ ]:
def compute_direction(layer: int) -> torch.Tensor:
    H, D = [], []

    for p in HEDGED_PROMPTS:
        H.append(get_last_token_hidden(p, layer).numpy())
    for p in DIRECT_PROMPTS:
        D.append(get_last_token_hidden(p, layer).numpy())

    direction = np.mean(H, axis=0) - np.mean(D, axis=0)
    direction = torch.tensor(direction, dtype=torch.float32)
    direction = direction / (direction.norm() + 1e-8)
    return direction

In [ ]:
def semantic_similarity(prompt: str, output: str) -> float:
    embs = embedder.encode([prompt, output], normalize_embeddings=True)
    return float(np.dot(embs[0], embs[1]))


STOPWORDS = set("""
a an the is are was were be to of and or in on for with as by from that this it
explain what why how
""".split())


def keyword_overlap(prompt: str, output: str) -> float:
    def clean(s):
        return set(
            w.strip(".,!?").lower()
            for w in s.split()
            if w.lower() not in STOPWORDS and len(w) > 2
        )

    p = clean(prompt)
    o = clean(output)
    if not p:
        return 0.0
    return len(p & o) / len(p)

In [ ]:
HEDGE_MARKERS = [
    # Explicit disclaimers
    "as an ai",
    "cannot guarantee",
    "no guarantee",
    "not a substitute",
    "for informational purposes",
    "this is not",

    # Caution / uncertainty framing
    "it depends",
    "depends on",
    "may vary",
    "can vary",
    "might vary",
    "context dependent",
    "context-dependent",
    "in some cases",
    "in many cases",
    "not always",

    # Softening / hedging language
    "generally",
    "generally speaking",
    "typically",
    "often",
    "usually",
    "in general",
    "to some extent",

    # Limitations & caveats
    "limitations",
    "has limitations",
    "there are limitations",
    "assumptions",
    "under certain assumptions",
    "edge cases",
    "trade-offs",

    # Contrast / qualification
    "however",
    "that said",
    "on the other hand",
    "while this is true",
    "although",

    # Safety / careful framing
    "important to note",
    "worth noting",
    "keep in mind",
    "be careful",
    "should be careful"
]

def behavior_metrics(prompt: str, output: str) -> Dict:
    output_l = output.lower()
    toks = tokenizer.encode(output)

    return {
        "n_tokens": len(toks),
        "hedge_marker_hits": sum(m in output_l for m in HEDGE_MARKERS),
        "semantic_sim": semantic_similarity(prompt, output),
        "keyword_overlap": keyword_overlap(prompt, output),
    }

In [ ]:
@torch.no_grad()
def generate(prompt: str, layer: int, alpha: float, direction: torch.Tensor) -> str:
    text = format_chat(prompt)
    enc = tokenizer(text, return_tensors="pt").to(cfg.device)

    handle = None
    if alpha > 0:
        block = model.model.layers[layer]
        d = direction.to(cfg.device)

        def hook_fn(module, inputs):
            hs = inputs[0]
            return (hs - alpha * d.to(hs.dtype),) + inputs[1:]

        handle = block.register_forward_pre_hook(hook_fn)

    gen_ids = model.generate(
        **enc,
        do_sample=False,
        max_new_tokens=cfg.max_new_tokens
    )

    if handle is not None:
        handle.remove()

    full = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
    prompt_decoded = tokenizer.decode(enc["input_ids"][0], skip_special_tokens=True)
    return full[len(prompt_decoded):].strip()

In [ ]:
def run_sweep(layers, alphas, out_dir="out_safe_steering"):
    os.makedirs(out_dir, exist_ok=True)

    summary = []

    for L in layers:
        direction = compute_direction(L)

        for a in alphas:
            rows = []

            for p in EVAL_PROMPTS:
                out = generate(p, L, a, direction)
                metrics = behavior_metrics(p, out)

                rows.append({
                    "layer": L,
                    "alpha": a,
                    "prompt": p,
                    "output": out,
                    **metrics
                })

            avg = {
                "layer": L,
                "alpha": a,
                "avg_hedge": np.mean([r["hedge_marker_hits"] for r in rows]),
                "avg_semantic_sim": np.mean([r["semantic_sim"] for r in rows]),
                "avg_keyword_overlap": np.mean([r["keyword_overlap"] for r in rows]),
            }

            summary.append(avg)

            with open(f"{out_dir}/logs_L{L}_a{a}.jsonl", "w") as f:
                for r in rows:
                    f.write(json.dumps(r) + "\n")

    with open(f"{out_dir}/summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print("Saved results to", out_dir)

In [ ]:
layers_to_try = [4, 6, 8, 10]
alphas_to_try = [0.0, 0.1, 0.5, 1.0, 2.0, 3.0, 4.5, 5.0]

run_sweep(layers_to_try, alphas_to_try)

References:

Turner et al., 2023
“Representation Engineering: A Top-Down Approach to AI Transparency”
https://arxiv.org/abs/2310.01405

Subramani et al., 2022
“Extracting Latent Steering Vectors from Pretrained Language Models”
https://arxiv.org/abs/2205.05124

Meng et al., 2022
“Locating and Editing Factual Associations in GPT” (ROME)
https://arxiv.org/abs/2202.05262

Rimsky et al., 2024
“Steering LLM Behavior with Activation Steering”
https://arxiv.org/abs/2402

Li et al., 2026
“Steering Vector Fields for Context-Aware Inference-Time Control in Large Language Models”
https://arxiv.org/abs/2602.01654